In [ ]:
from fasthtml.common import *
from fasthtml.jupyter import *
from fasthtml.components import (Section_Box, Note_Box, Eq_Box, Mat_Cell, Mat_Grid, Mat_Row_Label,
    Mat_Vecs, Mv_Vec, V_Dim, Dot_Cell, Stack_Mat, Stack_Row, Mix_Note, Mix_View, Mix_Side, Prop_Tag, Figure_Stub, Hover_Hint,
    App_Header, Brand_Link, Nav_Link, Color_Legend, Legend_Title, Legend_Item, Swatch, Hero_Band, Hero_Rule, Page_Main, Page_Footer)
from fasthtml.svg import Svg, Line, Circle, Text, G
import re, hashlib

In [ ]:
sent = "the cat sat on the mat".split()
n = len(sent)
imgs = ["cat","mat","dog","sun","car"]
ilbls = [f"image{i+1}" for i in range(5)]

def embed(tok, d=6): h = int(hashlib.md5(tok.encode()).hexdigest(),16); return [((h>>(i*8))&0xff)/255 for i in range(d)]
def wmat(seed, d=6): return [[(((int(hashlib.md5(f"{seed}-{i}-{j}".encode()).hexdigest(),16))&0xff)/255-.5) for j in range(d)] for i in range(d)]
def proj(vec, seed, d=6): return [sum(vec[r]*wmat(seed)[r][c] for r in range(d)) for c in range(d)]
def dot(a, b): return sum(x*y for x,y in zip(a,b))
def softmax(xs): m=max(xs); es=[2.718**(x-m) for x in xs]; s=sum(es); return [e/s for e in es]

Ws = dict(q=wmat("Wq"), k=wmat("Wk"), v=wmat("Wv"))
Q = {i:proj(embed(t),"q") for i,t in enumerate(sent)}
K = {i:proj(embed(t),"k") for i,t in enumerate(sent)}
V = {i:proj(embed(t),"v") for i,t in enumerate(sent)}
xs0,vs0 = embed("cat"), proj(embed("cat"),"v")

def Note_Md(*c): return Note_Box(*c)
def stub(label): return Figure_Stub(label)
def wgrid(seed): return Mat_Grid(*[Mat_Cell(f"{Ws[seed][r][c]:.2f}") for r in range(6) for c in range(6)])
def wgrid_c(seed, grp): return Mat_Grid(*[Mat_Cell(f"{Ws[seed][r][c]:.2f}", cls=f"sm {grp} c{c}") for r in range(6) for c in range(6)])
def umix(i): return [sum(V[j][d] for j in range(i+1))/(i+1) for d in range(6)]
def lbl_stack(rows, kind=None): return Stack_Mat(*[Stack_Row(Mat_Row_Label(lbl), *[V_Dim(f"{v[d]:.1f}", kind=kind) for d in range(6)]) for lbl,v in rows])
def mix_out(): return lbl_stack([(sent[i],umix(i)) for i in range(n)], kind="o")

def dense_net():
    ys = [40+i*40 for i in range(6)]; ix,ox = 90,360
    edges = [Line(x1=ix,y1=ys[r],x2=ox,y2=ys[c],stroke="#6366f1",stroke_width=1.3,opacity=round(.1+.9*abs(Ws['v'][r][c]),2),cls=f"e r{r} c{c}") for r in range(6) for c in range(6)]
    inn = [G(Circle(cx=ix,cy=y,r=15,fill="#e2e8f0"),Text(f"{xs0[i]:.1f}",x=ix,y=y+4,text_anchor="middle",font_size=10),cls=f"inp r{i}") for i,y in enumerate(ys)]
    out = [G(Circle(cx=ox,cy=y,r=15,fill="#fef3c7"),Text(f"{vs0[i]:.1f}",x=ox,y=y+4,text_anchor="middle",font_size=10),cls=f"o c{i}") for i,y in enumerate(ys)]
    return Div(Svg(*edges,*inn,*out,width=460,height=290), style="text-align:center")

def proj_eq_view():
    xr = Mv_Vec(*[V_Dim(f"{xs0[i]:.1f}", cls=f"minp r{i}") for i in range(6)])
    wg = Mat_Grid(*[Mat_Cell(f"{Ws['v'][r][c]:.2f}", cls=f"mw r{r} c{c}") for r in range(6) for c in range(6)])
    res = Mv_Vec(*[V_Dim(f"{vs0[i]:.1f}", kind="v", cls=f"mout c{i}") for i in range(6)])
    return Mat_Vecs(Mat_Row_Label("x"), xr, Dot_Cell("×"), Mat_Row_Label("W"), wg, Dot_Cell("="), res)

def in_stack(): return Stack_Mat(*[Stack_Row(Mat_Row_Label(ilbls[i]), *[V_Dim(f"{embed(t)[d]:.1f}", cls=f"sr img r{i}") for d in range(6)]) for i,t in enumerate(imgs)])
def out_stack(): return Stack_Mat(*[Stack_Row(Mat_Row_Label(ilbls[i]), *[V_Dim(f"{proj(embed(t),'v')[d]:.1f}", kind="v", cls=f"so img r{i} c{d}") for d in range(6)]) for i,t in enumerate(imgs)])
def in_sent(grp): return Stack_Mat(*[Stack_Row(Mat_Row_Label(sent[i]), *[V_Dim(f"{embed(sent[i])[d]:.1f}", cls=f"sr {grp} r{i}") for d in range(6)]) for i in range(n)])
def out_sent(): return Stack_Mat(*[Stack_Row(Mat_Row_Label(sent[i]), *[V_Dim(f"{V[i][d]:.1f}", kind="v", cls=f"so sent r{i} c{d}") for d in range(6)]) for i in range(n)])
def out_sent_kind(seed): return Stack_Mat(*[Stack_Row(Mat_Row_Label(sent[i]), *[V_Dim(f"{proj(embed(sent[i]),seed)[d]:.1f}", kind=seed, cls=f"so {seed}g r{i} c{d}") for d in range(6)]) for i in range(n)])
def v_side(): return Stack_Mat(*[Stack_Row(Prop_Tag("", cls=f"prop p{i}"), Mat_Row_Label(sent[i]), *[V_Dim(f"{V[i][d]:.1f}", kind="v") for d in range(6)], cls=f"vrow r{i}") for i in range(n)])
def qk_q(): return Stack_Mat(*[Stack_Row(Mat_Row_Label(sent[i]), *[V_Dim(f"{Q[i][d]:.1f}", kind="q", cls=f"qr r{i}") for d in range(6)]) for i in range(n)])
def qk_kt(): return Div(Div(*[Span(sent[j], cls="kthead") for j in range(n)], cls="kthead-row"), Mat_Grid(*[Mat_Cell(f"{K[j][d]:.1f}", cls=f"kt c{j}") for d in range(6) for j in range(n)], style=f"grid-template-columns:repeat({n}, 2.1rem)"))
def qk_a(): return Mat_Grid(*[Mat_Cell(f"{dot(Q[i],K[j]):.1f}", cls=f"av r{i} k{j}") for i in range(n) for j in range(n)], style=f"grid-template-columns:repeat({n}, 2.1rem)")
def masked_a(): return Mat_Grid(*[(Mat_Cell("−∞", cls="av masked") if j>i else Mat_Cell(f"{dot(Q[i],K[j]):.1f}", cls="av")) for i in range(n) for j in range(n)], style=f"grid-template-columns:repeat({n}, 2.1rem)")
def srow(i): sc=softmax([dot(Q[i],K[j]) for j in range(i+1)]); return sc+[0.0]*(n-i-1)
def softmax_a(): return Mat_Grid(*[(Mat_Cell("0", cls="av masked") if j>i else Mat_Cell(f"{srow(i)[j]:.2f}", cls="av", style=f"opacity:{0.3+0.7*srow(i)[j]/max(srow(i))}")) for i in range(n) for j in range(n)], style=f"grid-template-columns:repeat({n}, 2.1rem)")
def mixed_side(): return Stack_Mat(*[Stack_Row(Mat_Row_Label(sent[i]), *[V_Dim(f"{umix(i)[d]:.1f}", kind="o") for d in range(6)], cls=f"mrow r{i}", mi=i) for i in range(n)])
def asoft(i,j): return srow(i)[j] if j<=i else 0.0
def Oval(i,d): return sum(asoft(i,j)*V[j][d] for j in range(n))
def av_a(): return Mat_Grid(*[Mat_Cell(f"{asoft(i,j):.2f}", cls=f"oa r{i} k{j}") for i in range(n) for j in range(n)], style=f"grid-template-columns:repeat({n}, 2.1rem)")
def av_v(): return Stack_Mat(*[Stack_Row(Mat_Row_Label(sent[j]), *[V_Dim(f"{V[j][d]:.1f}", kind="v", cls=f"ov k{j}") for d in range(6)]) for j in range(n)])
def av_o(): return Stack_Mat(*[Stack_Row(Mat_Row_Label(sent[i]), *[V_Dim(f"{Oval(i,d):.1f}", kind="o", cls=f"oo r{i}") for d in range(6)]) for i in range(n)])
def aunif(i,j): return 1/(i+1) if j<=i else 0.0
def aunif_a(): return Mat_Grid(*[(Mat_Cell("0", cls=f"av masked ua r{i} k{j}") if j>i else Mat_Cell(f"{aunif(i,j):.2f}", cls=f"av ua r{i} k{j}")) for i in range(n) for j in range(n)], style=f"grid-template-columns:repeat({n}, 2.1rem)")
def uav_v(): return Stack_Mat(*[Stack_Row(Mat_Row_Label(sent[j]), *[V_Dim(f"{V[j][d]:.1f}", kind="v", cls=f"uv k{j}") for d in range(6)]) for j in range(n)])
def umix_o(): return Stack_Mat(*[Stack_Row(Mat_Row_Label(sent[i]), *[V_Dim(f"{umix(i)[d]:.1f}", kind="o", cls=f"uo r{i}") for d in range(6)]) for i in range(n)])

In [ ]:
def fig_for(s):
    s = s.lower()
    if 'core equation' in s: return Eq_Box(Strong("x' = Wx"))
    if 'matrix operations' in s: return proj_eq_view()
    if 'perceptron' in s: return dense_net()
    if 'label row' in s: return Mat_Vecs(Mat_Row_Label("X"), in_sent("sent"), Dot_Cell("×"), Mat_Row_Label("W"), wgrid_c("v","sent"), Dot_Cell("="), Mat_Row_Label("V"), out_sent())
    if 'uniform a' in s: return Mat_Vecs(Mat_Row_Label("A"), aunif_a(), Dot_Cell("×"), Mat_Row_Label("V"), uav_v(), Dot_Cell("="), Mat_Row_Label("mixed"), umix_o())
    if 'mixed' in s: return Mix_View(Mix_Side(Mat_Row_Label("V"), v_side()), Dot_Cell("→"), Mix_Side(Mat_Row_Label("mixed"), mixed_side()))
    if 'k and q' in s: return Div(Mat_Vecs(Mat_Row_Label("X"), in_sent("qg"), Dot_Cell("×"), Mat_Row_Label("Wq"), wgrid_c("q","qg"), Dot_Cell("="), Mat_Row_Label("Q"), out_sent_kind("q")), Mat_Vecs(Mat_Row_Label("X"), in_sent("kg"), Dot_Cell("×"), Mat_Row_Label("Wk"), wgrid_c("k","kg"), Dot_Cell("="), Mat_Row_Label("K"), out_sent_kind("k")), Mat_Vecs(Mat_Row_Label("Q"), qk_q(), Dot_Cell("×"), Mat_Row_Label("Kᵀ"), qk_kt(), Dot_Cell("="), Mat_Row_Label("A"), qk_a()))
    if 'top-right' in s: return Mat_Vecs(Mat_Row_Label("A"), masked_a())
    if 'softmax' in s: return Mat_Vecs(Mat_Row_Label("A"), softmax_a())
    if 'o being generated' in s: return Mat_Vecs(Mat_Row_Label("A"), av_a(), Dot_Cell("×"), Mat_Row_Label("V"), av_v(), Dot_Cell("="), Mat_Row_Label("O"), av_o())
    return Figure_Stub(s)

def build(txt):
    out = []
    for i,p in enumerate(re.split(r'\*([^*]+)\*', txt)):
        if i%2: out.append(fig_for(p))
        else: out += [Note_Box(q.strip()) for q in p.split('\n\n') if q.strip()]
    return Section_Box(*out)

When I took the Georgia Tech masters in CSci course on deep learning we briefly covered LLMs and how they are built by attention blocks. We trained a model in PyTorch and needed to manipuate some special Q, K, and V matrices the right way to achieve convergence. But it wasn't that intuitive and I find myself constantly forgetting how it works. Here is my opportunity to learn it again and distill it in a way where you could rediscover it by intuition whenever you need.

I hope you have some base information on neural networks. If not that's okay you will just have to accept one premise: NNs almost all involve operations where some input data x is matrix multiplied by a weight matrix W to create a new projection of x. 

*show the core equation*

For example, imagine we have $x$ represented by an image of 128 flattened numbers. If we multiply it by $W$ which might be $128\times256$ then we project that $x$ into a new vector of length $256$.

*show the matrix operations with highlighting*

Sometimes you will see this represented by a set of perceptrons densely conencted to each other.

*show the perceptron representation we should have hover highligthing for the output vaues highlighting all the conenctions but also highlighting the relevant values and stuff from the matrix number representation*

Normally we would apply some activation function nonlinearity at the end, but we will ignore this. Attention is able to bake in its nonlinearity differently.

If we want to compute more images simultaneously (likely on a GPU) we can feed more than one x into the weight matrix by just creating an X with more rows.
Note that adding a new x2 does not change the projection results from x1. They are independent as they should be. You wouldn't want the processing of one image to affect the result of the other.

But lets change the input data. Instead of X's rows being an unordered set of images, now they are an ordered set of tokens representing words.

*show the same image above except label row as "the cat sat on the mat" or something*

We will call this output matrix V, Not because it is particularily special, but just because it is a fine variable name consistent with the attention paper.

Now imagine we wanted to predict the next token like in a chatbot. Right now we process each token independently. If our model is to learn grammar, ordering, and complex nuance between words we need a way to intelligently mix these outputs.

Here is one concrete example: We could take V and set each row to be some normalized combination of the rows of V. For example the last row could be a uniform combination of all rows in V. Keep in mind though we are building a chatbot so we don't want to leak information from a future token to a past token. So our rule will be that we can only mix rows with itself and earlier rows. One consequence of this is that the first row will always be unchanged because it is a mixture only with itself.

*Show a transformation of the output matrix getting mixed. need some dynamic hover showing them getting mixed. We could even mix uniformly for demonstration.*

This uniform mixing is just one choice. Natrually, we will want the model to learn what the optimial distribution. To do this we will need to learn an $n\times n$ matrix called $A$, with $n$ being the size of the sentence.

*Show A times V producing the mixed matrix, using the uniform A from above*

This is how an attention module does it: Take $X$ and create two weight matrices $W_q$ and $W_k$. Compute $XW_q$ and $XW_k$ just like we did for $XW_v$. This will produce two new matrices $Q$ and $K$. Just like $V$, $Q$ and $K$ are just new, unique projections of $X$. Exactly the same as before. In the attention is all you need paper these are named for Key, Value, and Query. I found this more confusing than helpful. Just remember they are created the same way with the only difference being the different weights

To create an $n\times n$ matrix all we have to do is matrix multiply $Q$ by $K^\top$ (the transpose of $K$, so $n\times d$ times $d\times n$ gives an $n\times n$ result).

*Show K and Q being generated just liek above. Also show the matrix multiplication to produce the nxn matrix*

A matrix multiplication is just a lot of dot products, in this case dot products between every pair of $K$/$Q$ representations of our original tokens. The value of a dot product is higher the more similar a vector $k$ is to $q$. Therefore this $n\times n$ matrix is some measure of similiarty. Our hope is that as our model learns this measure of similarity becomes valuable.

But remember we only want information from future tokens to mix with past tokens, so first we mask; every pair where a token attends to a future token is set to $-\infty$, blanking the top-right of the matrix.

*show the future pairs in the top-right being set to −infinity*

Then we convert each row into probabilities with a row-wise softmax. The $-\infty$ entries become exactly zero, so each row sums to one.

*Show the infinity going through softmax porducing the final A matrix*

Now that we finally have our A matrix we can apply it to our V matrix and get the final result.

*show o being generated*

O now represents each token as a mixture of the ones in V. In one round of attention we found similarities between pairs of token projections. But now that we have these mixed token representations the more we pass them through attention layers the more they mix and the more likely we'll find interesting relationships fall out of our data.

By the way we didn't need a nonlinearity like a ReLu because we used a softmax during mixing. We also only needed to learn 3 weight matrices Wk, Wq, and Wv. (Actually you can have a Wo at the very end to project O one more time)

In general LLMs are just variations on stacking these modules in a very deep nearal network. You can stack them vertically, you can also do some tricks to stack them horizontally. You may wonder how we will predict the next token for our sentence. In practice you take the last row of the O matrix and send it through another weight matrix to predict the next token. But you can do other things too. You can translate, predict sentiment, or anything you think of. Notice how subtle architectural decisions matter for different use cases. For example our masking was useful for chatbot next token prediction, but if you were to translate a whole sentence it would make more sense to not mask at all because it is very helpful to mix past tokens with future ones.

We will see later that this architecture creates new challenges. In particular when new tokens are generated the size of the K/V matrices will grow leading to new challenges for training and inference.

In [ ]:
txt = (await read_msgid('_04663a93'))['content']

In [ ]:
theme_css = Style("""
*{box-sizing:border-box;}
body{margin:0;background:#FCFBF8;color:#3A352B;font-family:'Hanken Grotesk',sans-serif;font-size:17px;line-height:1.72;-webkit-font-smoothing:antialiased;}
app-header{position:sticky;top:0;z-index:50;display:flex;align-items:center;justify-content:space-between;height:54px;padding:0 24px;background:rgba(252,251,248,.86);backdrop-filter:blur(10px);border-bottom:1px solid #E8E4DB;}
brand-link a{font:600 15px/1 'Hanken Grotesk',sans-serif;color:#221C12;text-decoration:none;}
nav-link a{font:500 13.5px/1 'Hanken Grotesk',sans-serif;color:#5B4FBF;text-decoration:none;}
color-legend{position:sticky;top:54px;z-index:49;display:flex;gap:18px;align-items:center;padding:9px 24px;background:rgba(252,251,248,.93);backdrop-filter:blur(10px);border-bottom:1px solid #E8E4DB;overflow-x:auto;}
legend-title{font:600 10px/1 'Hanken Grotesk',sans-serif;letter-spacing:.08em;text-transform:uppercase;color:#A39B8B;white-space:nowrap;}
legend-item{display:inline-flex;align-items:center;gap:6px;white-space:nowrap;font:500 12px/1 'Hanken Grotesk',sans-serif;color:#5A5346;}
legend-item swatch{display:block;width:13px;height:13px;border-radius:4px;border:1px solid rgba(0,0,0,.08);}
legend-item strong{font-family:'Newsreader',serif;font-style:italic;color:#403A32;}
hero-band{display:block;max-width:760px;margin:0 auto;padding:62px 24px 6px;text-align:center;}
hero-band h1{font-family:'Newsreader',serif;font-weight:500;font-size:clamp(2.5rem,6.5vw,3.7rem);line-height:1.04;letter-spacing:-.012em;color:#1E1B14;margin:0;}
hero-rule{display:block;max-width:560px;margin:40px auto 8px;height:1px;background:#E8E4DB;}
page-main{display:block;max-width:1040px;margin:0 auto;padding:14px 20px 40px;}
section-box{display:block;max-width:1040px;margin:0 auto 10px;}
note-box{display:block;max-width:660px;margin:0 auto 1.05rem;text-align:justify;}
note-box strong{font-weight:600;color:#221C12;}
eq-box{display:block;max-width:520px;margin:18px auto 22px;text-align:center;background:#F3F1EA;border:1px solid #E8E4DB;border-radius:12px;padding:14px;font-family:'Newsreader',serif;font-style:italic;font-size:1.5rem;color:#2A2620;}
mat-vecs{display:flex;align-items:center;gap:.65rem;flex-wrap:nowrap;overflow-x:auto;justify-content:safe center;background:#fff;border:1px solid #E8E4DB;border-radius:16px;padding:22px 18px;margin:24px auto;max-width:1000px;box-shadow:0 1px 3px rgba(40,33,24,.05);}
mv-vec{display:inline-flex;gap:3px;}
v-dim{display:block;min-width:2.25rem;padding:.34rem .2rem;text-align:center;font-family:'IBM Plex Mono',monospace;font-size:.72rem;background:#E9ECF2;color:#59616F;border-radius:5px;}
v-dim[kind="v"]{background:#FAEBC8;color:#8A5A12;}
v-dim[kind="o"]{background:#EBE0FB;color:#6B3FA8;}
v-dim[kind="q"]{background:#DBE5FA;color:#2B4FA0;}
v-dim[kind="k"]{background:#D6EEDB;color:#1E7A43;}
mat-row-label{display:block;font-family:'Newsreader',serif;font-style:italic;font-weight:500;color:#403A32;min-width:1.5rem;text-align:right;}
dot-cell{display:block;font-family:'Newsreader',serif;font-style:italic;font-size:1.15rem;color:#B0A99B;}
mat-grid{display:inline-grid;grid-template-columns:repeat(6,2.25rem);gap:3px;align-items:center;}
mat-cell{padding:.34rem .2rem;text-align:center;font-family:'IBM Plex Mono',monospace;font-size:.7rem;background:#F0EBE0;color:#86744F;border-radius:5px;}
stack-mat{display:inline-flex;flex-direction:column;gap:3px;}
stack-row{display:flex;gap:3px;align-items:center;}
mix-note{display:block;max-width:660px;margin:1rem auto;padding:.6rem .9rem;background:#F3F1EA;border-left:4px solid #5B4FBF;border-radius:.25rem;font-size:.9rem;}
mix-view{display:flex;align-items:flex-start;gap:1.5rem;justify-content:safe center;overflow-x:auto;background:#fff;border:1px solid #E8E4DB;border-radius:16px;padding:22px 18px;margin:24px auto;max-width:1000px;box-shadow:0 1px 3px rgba(40,33,24,.05);}
mix-side{display:inline-flex;flex-direction:column;gap:.3rem;}
prop-tag{display:inline-block;min-width:2.3rem;font-family:'IBM Plex Mono',monospace;font-size:.64rem;font-weight:500;color:#5B4FBF;text-align:right;padding-right:4px;}
.kthead-row{display:grid;grid-template-columns:repeat(6,2.1rem);gap:3px;margin-bottom:2px;}
.kthead{font:.55rem/1 'Hanken Grotesk',sans-serif;text-align:center;color:#8A8275;overflow:hidden;text-overflow:ellipsis;}
.av{background:#FADFEA;color:#AE3A6C;}
.av.masked{background:#EFEDE7;color:#B3AC9E;}
.vrow{opacity:.4;transition:opacity .15s;}.vrow.hl{opacity:1;}
.mrow{cursor:pointer;}
.sr.hl,.sm.hl,.qr.hl,.kt.hl,.av.hl,.oo.hl,.oa.hl,.ov.hl,.so.hl,.ua.hl,.uv.hl,.uo.hl,.mat-cell.hl{outline:2px solid currentColor;outline-offset:-1px;}
.e.hl{stroke:#5B4FBF;stroke-width:3;}
.inp.hl circle{fill:#C9D2E8;}
.o.hl circle{fill:#F3DFA0;}
.hover-hint{display:inline-block;margin-bottom:.5rem;padding:.25rem .7rem;background:#5B4FBF;color:#fff;border-radius:1rem;font-size:.78rem;transition:opacity .3s;}
.hover-hint.gone{opacity:0;}
.float-hint{position:absolute;left:calc(100% + 12px);top:50%;transform:translateY(-50%);white-space:nowrap;}
.mrow.hl v-dim{outline:2px solid #8A5A12;}
.minp.hl,.mw.hl,.mout.hl{outline:2px solid currentColor;outline-offset:-1px;}
@keyframes pulseBorder{0%,100%{box-shadow:0 0 0 0 rgba(91,79,191,0);}50%{box-shadow:0 0 0 3px rgba(91,79,191,.5);}}
.mout{animation:pulseBorder 1.5s ease-in-out infinite;cursor:default;}
.mout.seen{animation:none;}
page-footer{display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:10px;max-width:920px;margin:30px auto 0;padding:24px;border-top:1px solid #E8E4DB;font:400 13px/1.5 'Hanken Grotesk',sans-serif;color:#9A9285;}
page-footer a{color:#5B4FBF;text-decoration:none;}
""")

font_hdrs = (Link(rel="preconnect", href="https://fonts.googleapis.com"),
    Link(rel="preconnect", href="https://fonts.gstatic.com", crossorigin=""),
    Link(rel="stylesheet", href="https://fonts.googleapis.com/css2?family=Hanken+Grotesk:wght@400;500;600;700&family=IBM+Plex+Mono:wght@400;500&family=Newsreader:ital,wght@0,400;0,500;1,400;1,500&display=swap"))

mathjax = (Script("window.MathJax={tex:{inlineMath:[['$','$']],displayMath:[['$$','$$']]}};"),
    Script(src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js"))

In [ ]:
hl_js = Script("""
 const SENT = ['the', 'cat', 'sat', 'on', 'the', 'mat'];
document.addEventListener('mouseover', e => {
  let o = e.target.closest('.o');
  if (o) {
    let c = [...o.classList].find(x=>x.startsWith('c')).slice(1);
    document.querySelectorAll(`.e.c${c}`).forEach(x=>x.classList.add('hl'));
    document.querySelectorAll(`.m.c${c}`).forEach(x=>x.classList.add('hl'));
    document.querySelectorAll('.inp').forEach(x=>x.classList.add('hl'));
    o.classList.add('hl');
      let h=document.getElementById('hint'); if(h) h.classList.add('gone');
  }
    let mo = e.target.closest('.mout');
    if (mo) {
      let c = [...mo.classList].find(x=>/^c\\d/.test(x)).slice(1);
      document.querySelectorAll(`.mw.c${c}`).forEach(x=>x.classList.add('hl'));
      document.querySelectorAll('.minp').forEach(x=>x.classList.add('hl'));
      mo.classList.add('hl');
        document.querySelectorAll(`.o.c${c}`).forEach(x=>x.classList.add('hl'));
          document.querySelectorAll(`.e.c${c}`).forEach(x=>x.classList.add('hl'));
        document.querySelectorAll('.mout').forEach(x=>x.classList.add('seen'));
    }
  let s = e.target.closest('.so');
  if (s) {
    let r = [...s.classList].find(x=>/^r\\d/.test(x)).slice(1);
      let g = ['img','sent','qg','kg'].find(x=>s.classList.contains(x));
        let ig = g;
      let c = [...s.classList].find(x=>/^c\\d/.test(x)).slice(1);
      document.querySelectorAll(`.sr.${ig}.r${r}`).forEach(x=>x.classList.add('hl'));
      document.querySelectorAll(`.sm.${g}.c${c}`).forEach(x=>x.classList.add('hl'));
    s.classList.add('hl');
  }
  let m = e.target.closest('.mrow');
  if (m) {
    let i = +m.getAttribute('mi'); m.classList.add('hl');
    let p = (1/(i+1)).toFixed(2);
    for (let j=0;j<=i;j++){
      document.querySelector(`.vrow.r${j}`).classList.add('hl');
      document.querySelector(`.vrow.r${j} .prop`).textContent = p;
    }
  }
    let a = e.target.closest('.av');
    if (a) {
      let r = [...a.classList].find(x=>/^r\\d/.test(x)).slice(1);
      let k = [...a.classList].find(x=>/^k\\d/.test(x)).slice(1);
      document.querySelectorAll(`.qr.r${r}`).forEach(x=>x.classList.add('hl'));
      document.querySelectorAll(`.kt.c${k}`).forEach(x=>x.classList.add('hl'));
      a.classList.add('hl');
        document.getElementById('aktip').textContent = `This is the similarity between ${SENT[r]}_q and ${SENT[k]}_k.`;
    }
    let oo = e.target.closest('.oo');
    if (oo) {
      let r = [...oo.classList].find(x=>/^r\\d/.test(x)).slice(1);
      document.querySelectorAll(`.oo.r${r}`).forEach(x=>x.classList.add('hl'));
      document.querySelectorAll(`.oa.r${r}`).forEach(x=>x.classList.add('hl'));
        for (let j=0;j<=+r;j++) document.querySelectorAll(`.ov.k${j}`).forEach(x=>x.classList.add('hl'));
    }
    let uo = e.target.closest('.uo');
    if (uo) {
      let r = [...uo.classList].find(x=>/^r\\d/.test(x)).slice(1);
      document.querySelectorAll(`.uo.r${r}`).forEach(x=>x.classList.add('hl'));
      document.querySelectorAll(`.ua.r${r}`).forEach(x=>x.classList.add('hl'));
        for (let j=0;j<=+r;j++) document.querySelectorAll(`.uv.k${j}`).forEach(x=>x.classList.add('hl'));
    }
});
document.addEventListener('mouseout', e => {
    if (e.target.closest('.o') || e.target.closest('.mout') || e.target.closest('.so') || e.target.closest('.mrow') || e.target.closest('.av') || e.target.closest('.oo') || e.target.closest('.uo')) {
    document.querySelectorAll('.hl').forEach(x=>x.classList.remove('hl'));
    document.querySelectorAll('.prop').forEach(x=>x.textContent='');
  }
});
""")

In [ ]:
legend = [("X","input","#E9ECF2"),("W","weights","#F0EBE0"),("V","value","#FAEBC8"),("Q","query","#DBE5FA"),("K","key","#D6EEDB"),("A","attention","#FADFEA"),("O","output","#EBE0FB")]
def legend_item(sym, name, bg): return Legend_Item(Swatch(style=f"background:{bg}"), Span(Strong(sym), f" {name}"))
def color_legend(): return Color_Legend(Legend_Title("colour key"), *[legend_item(*x) for x in legend])
def chrome_header(): return App_Header(Brand_Link(A("How attention works", href="#")), Nav_Link(A("Lisa Dlima ↗", href="https://lisadlima.com", target="_blank")))
def hero(): return Hero_Band(H1("How attention works"), Span("By ", A("Lisa Dlima", href="https://lisadlima.com", target="_blank")))
def page_footer(): return Page_Footer(Span("An interactive walkthrough by ", A("Lisa Dlima", href="https://lisadlima.com", target="_blank")), A("Attention Is All You Need ↗", href="https://arxiv.org/abs/1706.03762", target="_blank"))

In [ ]:
seo = (Meta(name="description", content="An interactive, visual walkthrough of how attention works in transformers."),
    Link(rel="canonical", href="https://lisadlima.com/attention"),
    Meta(property="og:title", content="How attention works"),
    Meta(property="og:site_name", content="lisadlima.com"),
    Meta(property="og:description", content="An interactive visual walkthrough of attention in transformers."),
    Meta(property="og:type", content="article"),
    Meta(property="og:url", content="https://lisadlima.com/attention"),
    Meta(property="og:image", content="https://lisadlima.com/card.png"),
    Meta(property="og:image:width", content="1200"),
    Meta(property="og:image:height", content="630"),
    Meta(name="twitter:card", content="summary_large_image"))

app, rt = fast_app(pico=False, title="How attention works", canonical=False, hdrs=(*font_hdrs, theme_css, hl_js, mathjax, *seo))

@rt('/')
def index(): return Div(chrome_header(), color_legend(), hero(), Hero_Rule(), Page_Main(build(txt)), page_footer())

In [ ]:
srv = JupyUvi(app, port=8001)

In [ ]:
from PIL import Image, ImageDraw, ImageFont

W,H = 1200,630
toks = "The cat sat on the mat".split()
gx,gy,cell = 760,170,56

def circle_crop(im, d):
    s = min(im.size); im = im.crop(((im.width-s)//2,(im.height-s)//2,(im.width+s)//2,(im.height+s)//2)).resize((d,d), Image.LANCZOS)
    mask = Image.new('L',(d,d),0); ImageDraw.Draw(mask).ellipse((0,0,d,d),fill=255)
    out = Image.new('RGBA',(d,d),(0,0,0,0)); out.paste(im,(0,0),mask); return out

def shade(w, wmax): return tuple(round(c+(250-c)*(1-(.3+.7*w/wmax))) for c in (174,58,108))
def att_grid(draw, x0, y0, cell, rows):
    for i in range(rows):
        for j in range(rows): draw.rectangle((x0+j*cell,y0+i*cell,x0+j*cell+cell-5,y0+i*cell+cell-5), fill='#EFEDE7' if j>i else shade(1/(i+1), 1/(i+1)))

ft = lambda f,s: ImageFont.truetype(f'/usr/share/fonts/truetype/{f}', s)
serif,sans = ft('liberation2/LiberationSerif-Regular.ttf',96), ft('lato/Lato-Semibold.ttf',30)
sub,lbl,brand = ft('lato/Lato-Regular.ttf',28), ft('lato/Lato-Medium.ttf',22), ft('lato/Lato-Regular.ttf',26)

avatar = circle_crop(Image.open('lisa_pool.jpg'), 90)
card = Image.new('RGB', (W,H), '#FCFBF8')
draw = ImageDraw.Draw(card)
draw.text((64,120), "How", font=serif, fill='#1E1B14')
draw.text((64,220), "attention", font=serif, fill='#1E1B14')
draw.text((64,320), "works", font=serif, fill='#5B4FBF')
draw.text((66,432), "an interactive, visual walkthrough", font=sub, fill='#5A5346')
att_grid(draw, gx, gy, cell, 6)
for j,t in enumerate(toks): draw.text((gx+j*cell+(cell-5)//2, gy-16), t, font=lbl, fill='#86744F', anchor='mb')
for i,t in enumerate(toks): draw.text((gx-14, gy+i*cell+(cell-5)//2), t, font=lbl, fill='#86744F', anchor='rm')
card.paste(avatar, (64,500), avatar)
draw.text((170,522), "Lisa Dlima", font=sans, fill='#403A32')
draw.text((170,558), "lisadlima.com", font=brand, fill='#A39B8B')
card.save('card.png'); card.save('../card.png')
card